# AI training energy & carbon: full pipeline
Runs every stage on the frozen inputs dataset. Seed 42 everywhere. Outputs go to `/kaggle/working/results/` (tables + figures) and `/kaggle/working/processed/`.

## 0. Project layout, integrity check, packages

In [ ]:

import os, shutil, subprocess, sys, pathlib
INP = pathlib.Path("/kaggle/input")
def find(name):
    hits = sorted(INP.rglob(name))
    assert hits, f"{name} not found under /kaggle/input"
    return hits[0]
PROJ = pathlib.Path("/kaggle/working/proj")
if PROJ.exists(): shutil.rmtree(PROJ)
(PROJ / "data").mkdir(parents=True)
shutil.copytree(find("energy.py").parent, PROJ / "src")
shutil.copytree(find("test_energy.py").parent, PROJ / "tests")
raw = PROJ / "data/raw"
for rel in ["epoch/ai_models/notable_ai_models.csv", "epoch/ai_models/frontier_ai_models.csv",
            "epoch/ml_hardware/ml_hardware.csv", "owid/carbon-intensity-electricity.csv",
            "ember/release_generation_yearly_global.csv", "download_log.csv"]:
    (raw / rel).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(find(pathlib.Path(rel).name), raw / rel)
(PROJ / "data/validation").mkdir(parents=True, exist_ok=True)
for name in ["reported_energy.csv", "measured_system_power.csv"]:
    hits = sorted(INP.rglob(name))
    if hits:
        shutil.copy(hits[0], PROJ / "data/validation" / name)
os.chdir(PROJ)
print(sorted(str(p.relative_to(PROJ)) for p in PROJ.rglob("*") if p.is_file()))


In [ ]:

# integrity: compare SHA-256 of the frozen raw files with the download log
import hashlib, pandas as pd
log = pd.read_csv("data/raw/download_log.csv")
for _, r in log.iterrows():
    p = pathlib.Path("data/raw") / r.file
    if p.exists():
        print(("OK       " if hashlib.sha256(p.read_bytes()).hexdigest() == r.sha256 else "MISMATCH ") + r.file)


In [ ]:

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "SALib"], check=False)
import numpy, pandas, sklearn, shap, SALib, xgboost, lightgbm
print({m.__name__: m.__version__ for m in [numpy, pandas, sklearn, shap, xgboost, lightgbm]}, sys.version)


## 1. Unit tests for the energy formulas

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"], capture_output=True, text=True)
print(r.stdout[-3000:]); assert r.returncode == 0

## 2. Data preparation and coverage

In [ ]:
r = subprocess.run([sys.executable, "-P", "-W", "ignore", "src/prepare.py"], capture_output=True, text=True)
print(r.stdout[-20000:]); print(r.stderr[-5000:]); assert r.returncode == 0

## 3. Monte Carlo estimation, Method A vs B, Epoch comparison, sensitivity

In [ ]:
r = subprocess.run([sys.executable, "-P", "-W", "ignore", "src/estimate.py"], capture_output=True, text=True)
print(r.stdout[-20000:]); print(r.stderr[-5000:]); assert r.returncode == 0

## 4. Validation against reported values

In [ ]:
if pathlib.Path("src/validate.py").exists() and pathlib.Path("data/validation/reported_energy.csv").exists():
    r = subprocess.run([sys.executable, "-P", "-W", "ignore", "src/validate.py"], capture_output=True, text=True)
    print(r.stdout[-20000:]); print(r.stderr[-5000:]); assert r.returncode == 0
else:
    print("validation set not present in this input version")

## 5. Trends, doubling times, 2030 scenarios, location experiment

In [ ]:
r = subprocess.run([sys.executable, "-P", "-W", "ignore", "src/trends.py"], capture_output=True, text=True)
print(r.stdout[-20000:]); print(r.stderr[-5000:]); assert r.returncode == 0

## 6. ML imputation with SHAP

In [ ]:
r = subprocess.run([sys.executable, "-P", "-W", "ignore", "src/ml.py"], capture_output=True, text=True)
print(r.stdout[-20000:]); print(r.stderr[-5000:]); assert r.returncode == 0

## 7. Figures

In [ ]:
r = subprocess.run([sys.executable, "-P", "-W", "ignore", "src/figures.py"], capture_output=True, text=True)
print(r.stdout[-20000:]); print(r.stderr[-5000:]); assert r.returncode == 0

In [ ]:
from IPython.display import Image, display
for p in sorted(pathlib.Path("results/figures").glob("*.png")):
    print(p.name); display(Image(str(p), width=900))

## 8. Export

In [ ]:
shutil.copytree("results", "/kaggle/working/results", dirs_exist_ok=True)
shutil.copytree("data/processed", "/kaggle/working/processed", dirs_exist_ok=True)
os.chdir("/kaggle/working"); shutil.rmtree(PROJ)
print(sorted(str(p) for p in pathlib.Path("/kaggle/working").rglob("*") if p.is_file()))